In [1]:
import os
import sys
import json
import textwrap
import subprocess
from pathlib import Path
from datetime import datetime

# --- repo + run configuration ---------------------------------------------

REPO_ROOT = Path(".").resolve()          # assume notebook at repo root
RUN_ID    = "1760006816"                 # your trained run
RUN_DIR   = REPO_ROOT / "notebooks" / "out" / "demo_case" / RUN_ID

MODEL_PATH = RUN_DIR / "demo_gcbert.pt"
CFG_PATH   = RUN_DIR / "gcbert_gnn_attn.cfg.json"   # adjust if stored elsewhere

# where we will put this new multi-file demo under that run
DEMO_OUT_DIR = RUN_DIR / "multifile_demo"
DEMO_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT :", REPO_ROOT)
print("RUN_DIR   :", RUN_DIR)
print("MODEL     :", MODEL_PATH, " exists:", MODEL_PATH.is_file())
print("CFG       :", CFG_PATH,   " exists:", CFG_PATH.is_file())
print("DEMO_OUT  :", DEMO_OUT_DIR)

# --- source project + graph dirs ------------------------------------------

DEMO_ROOT        = DEMO_OUT_DIR / "project"
DEMO_APP_DIR     = DEMO_ROOT / "app"
DEMO_LIB_DIR     = DEMO_ROOT / "lib"

PDG_DIR          = DEMO_OUT_DIR / "pdg_raw"
CLEAN_PDG_DIR    = DEMO_OUT_DIR / "pdg_clean"
SLICES_DIR       = DEMO_OUT_DIR / "slices"
BAD_PDG_DIR      = DEMO_OUT_DIR / "pdg_bad"
CPG_BIN          = DEMO_OUT_DIR / "cpg.bin"

# --- tooling (adapt if different) -----------------------------------------

JOERN_DIR        = REPO_ROOT / "external" / "joern"
JOERN_C2CPG      = JOERN_DIR / "c2cpg.bat"
JOERN_MAIN       = JOERN_DIR / "joern.bat"

EXPORT_SCRIPT    = REPO_ROOT / "tools" / "export_pdg_env.sc"
SENSI_FILE       = REPO_ROOT / "src" / "preprocess" / "external" / "sensiAPI.txt"

print("JOERN_C2CPG:", JOERN_C2CPG)
print("JOERN_MAIN :", JOERN_MAIN)
print("EXPORT_SC  :", EXPORT_SCRIPT)
print("SENSI_FILE :", SENSI_FILE)


REPO_ROOT : C:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\notebooks
RUN_DIR   : C:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\notebooks\notebooks\out\demo_case\1760006816
MODEL     : C:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\notebooks\notebooks\out\demo_case\1760006816\demo_gcbert.pt  exists: False
CFG       : C:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\notebooks\notebooks\out\demo_case\1760006816\gcbert_gnn_attn.cfg.json  exists: False
DEMO_OUT  : C:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\notebooks\notebooks\out\demo_case\1760006816\multifile_demo
JOERN_C2CPG: C:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\notebooks\external\joern\c2cpg.bat
JOERN_MAIN : C:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\notebooks\external\joern\joern.bat
EXPORT_SC  : C:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\notebooks\tools\export_pdg_env.sc
SENSI_FILE : C:\Users\MSHUVO23\Desktop\The

In [3]:
# Create directory structure
for d in [
    DEMO_APP_DIR,
    DEMO_LIB_DIR,
    PDG_DIR,
    CLEAN_PDG_DIR,
    SLICES_DIR,
    BAD_PDG_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# Multi-file C code: ROOT in main.c, propagation in wrap.c, SINK in sink.c
files = {
    DEMO_APP_DIR / "main.c": r'''
        #include <stdio.h>
        #include <stdlib.h>
        #include <string.h>

        extern char* wrap_a(char* in);
        extern char* wrap_b(char* in);
        extern char* wrap_c(char* in);
        extern void run_system(char* cmd);

        int main(int argc, char** argv) {
            char buf[256];
            // ROOT: untrusted input (simulate reading from argv)
            if (argc > 1) {
                strncpy(buf, argv[1], sizeof(buf)-1);
                buf[sizeof(buf)-1] = '\0';
                char* p = wrap_a(buf);
                p = wrap_b(p);
                p = wrap_c(p);
                // final call: dangerous sink
                run_system(p);
            }
            return 0;
        }
    ''',
    DEMO_LIB_DIR / "wrap.c": r'''
        #include <stdio.h>
        #include <stdlib.h>
        #include <string.h>

        char tmp1[300];
        char tmp2[300];
        char tmp3[300];

        char* wrap_a(char* in) {
            // propagation #1 (DFG + CALL/ARG2PARAM)
            snprintf(tmp1, sizeof(tmp1), "%s", in);
            return tmp1;
        }

        char* wrap_b(char* in) {
            // propagation #2
            strcat(tmp2, in);
            strcat(tmp2, " --verbose");
            return tmp2;
        }

        char* wrap_c(char* in) {
            // propagation #3
            sprintf(tmp3, "%s %s", in, "--force");
            return tmp3;
        }
    ''',
    DEMO_LIB_DIR / "sink.c": r'''
        #include <stdio.h>
        #include <stdlib.h>
        #include <string.h>

        void run_system(char* cmd) {
            // SINK: vulnerable external command execution
            // (intentionally no sanitization)
            system(cmd);
        }
    ''',
}

for path, src in files.items():
    path.write_text(textwrap.dedent(src).strip() + "\n", encoding="utf-8")
    print("Wrote", path.relative_to(REPO_ROOT))

print("\nProject tree:")
for p in DEMO_ROOT.rglob("*.c"):
    print(" ", p.relative_to(REPO_ROOT))


Wrote notebooks\out\demo_case\1760006816\multifile_demo\project\app\main.c
Wrote notebooks\out\demo_case\1760006816\multifile_demo\project\lib\wrap.c
Wrote notebooks\out\demo_case\1760006816\multifile_demo\project\lib\sink.c

Project tree:
  notebooks\out\demo_case\1760006816\multifile_demo\project\app\main.c
  notebooks\out\demo_case\1760006816\multifile_demo\project\lib\sink.c
  notebooks\out\demo_case\1760006816\multifile_demo\project\lib\wrap.c


In [4]:
assert JOERN_C2CPG.is_file(),  "c2cpg.bat not found; fix JOERN_DIR"
assert JOERN_MAIN.is_file(),   "joern.bat not found; fix JOERN_DIR"
assert EXPORT_SCRIPT.is_file(),"export_pdg_env.sc not found"
assert SENSI_FILE.is_file(),   "sensiAPI.txt not found"

# 1) Build CPG for the whole small project
print("Running c2cpg on", DEMO_ROOT)
subprocess.run(
    [str(JOERN_C2CPG), "-J-Xmx4g", str(DEMO_ROOT), "--output", str(CPG_BIN)],
    check=True
)
print("CPG created:", CPG_BIN, "exists:", CPG_BIN.is_file())

# 2) Export PDGs
env = os.environ.copy()
env["CPG_PATH"]   = str(CPG_BIN.resolve())
env["OUT_DIR"]    = str(PDG_DIR.resolve())
env["SENSI_PATH"] = str(SENSI_FILE.resolve())
env["LOG_PATH"]   = str((PDG_DIR / "export_demo.log").resolve())

print("\nExporting PDGs to", PDG_DIR)
subprocess.run(
    [str(JOERN_MAIN), "--script", str(EXPORT_SCRIPT)],
    env=env,
    check=True
)

workspace = REPO_ROOT / "workspace"
if workspace.exists():
    import shutil
    shutil.rmtree(workspace)
    print("Removed Joern workspace")

print("\nPDG JSON files:")
for p in PDG_DIR.rglob("*.json"):
    print(" ", p.relative_to(REPO_ROOT))


AssertionError: c2cpg.bat not found; fix JOERN_DIR

In [ ]:
print("Running step2c_pipeline_validate_and_slice on demo PDGs")

cmd = [
    sys.executable, "-m", "src.preprocess.step2c_pipeline_validate_and_slice",
    "--src",   str(PDG_DIR),
    "--clean", str(CLEAN_PDG_DIR),
    "--out",   str(SLICES_DIR),
    "--bad",   str(BAD_PDG_DIR),
    "--sensi", str(SENSI_FILE),
    "--require_ddg",
    "--copy_bad",
]

print("Command:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("\nSlice files:")
slice_files = sorted(SLICES_DIR.glob("*.slice.json"))
for p in slice_files:
    print(" ", p.relative_to(REPO_ROOT))

assert slice_files, "No slice JSON produced; check previous steps"

SLICE_JSON = slice_files[0]
print("\nUsing slice:", SLICE_JSON.relative_to(REPO_ROOT))


In [ ]:
def list_new_files(root: Path, suffix):
    return {p.resolve() for p in root.rglob(f"*{suffix}")}

before_png = list_new_files(DEMO_OUT_DIR, ".png")
before_svg = list_new_files(DEMO_OUT_DIR, ".svg")
before_json = list_new_files(DEMO_OUT_DIR, ".json")
before_txt = list_new_files(DEMO_OUT_DIR, ".txt")

print("Running inference with trained model on multi-file slice")

cmd = [
    sys.executable, "-u", "-m", "src.infer_one_slice_pretty",
    "--slice_json", str(SLICE_JSON),
    "--model",      str(MODEL_PATH),
    "--cfg",        str(CFG_PATH),
    "--sensi_file", str(SENSI_FILE),
    "--max_len",    "16",
    "--print_topk", "16",
    "--hide_trivial_topn",
    "--min_attn",   "0.03",
    "--thr",        "0.9",
    "--hf_local_only",
]

result = subprocess.run(cmd, check=True, capture_output=True, text=True)
print("=== Raw stdout from inference ===")
print(result.stdout)

CHAIN_TXT = DEMO_OUT_DIR / "chain.txt"
CHAIN_TXT.write_text(result.stdout, encoding="utf-8")
print("\nSaved chain text to:", CHAIN_TXT)

after_png = list_new_files(DEMO_OUT_DIR, ".png")
after_svg = list_new_files(DEMO_OUT_DIR, ".svg")
after_json = list_new_files(DEMO_OUT_DIR, ".json")
after_txt = list_new_files(DEMO_OUT_DIR, ".txt")

new_png  = sorted(after_png - before_png, key=os.path.getmtime)
new_svg  = sorted(after_svg - before_svg, key=os.path.getmtime)
new_json = sorted(after_json - before_json, key=os.path.getmtime)

CHAIN_PNG  = new_png[-1]  if new_png  else None
CHAIN_SVG  = new_svg[-1]  if new_svg  else None
CHAIN_JSON = None

for cand in new_json:
    if "chain" in cand.name.lower():
        CHAIN_JSON = cand
        break

if CHAIN_JSON is None:
    lines = [ln.strip() for ln in result.stdout.splitlines() if ln.strip()]
    chain_nodes = [ln for ln in lines if ":" in ln]
    CHAIN_JSON = DEMO_OUT_DIR / "chain.json"
    with CHAIN_JSON.open("w", encoding="utf-8") as f:
        json.dump({"nodes": chain_nodes}, f, indent=2)
    print("Created minimal chain.json at", CHAIN_JSON)
else:
    print("Using existing chain.json at", CHAIN_JSON)

print("CHAIN_PNG:", CHAIN_PNG)
print("CHAIN_SVG:", CHAIN_SVG)

SOURCE_FILES_JSON = DEMO_OUT_DIR / "source_files.json"
src_map = {
    "app/main.c":  (DEMO_APP_DIR / "main.c").read_text(),
    "lib/wrap.c":  (DEMO_LIB_DIR / "wrap.c").read_text(),
    "lib/sink.c":  (DEMO_LIB_DIR / "sink.c").read_text(),
}
with SOURCE_FILES_JSON.open("w", encoding="utf-8") as f:
    json.dump(src_map, f, indent=2)
print("Saved source_files.json at", SOURCE_FILES_JSON)


In [ ]:
from IPython.display import Image, display

print("=== Source files ===\n")

for path in [DEMO_APP_DIR / "main.c", DEMO_LIB_DIR / "wrap.c", DEMO_LIB_DIR / "sink.c"]:
    print(f"--- {path.relative_to(REPO_ROOT)} ---")
    print(path.read_text())
    print()

print("\n=== Causal chain text (from inference) ===\n")
print(CHAIN_TXT.read_text())

print("\n=== Causal chain JSON path ===")
print(CHAIN_JSON.relative_to(REPO_ROOT))

if CHAIN_PNG and Path(CHAIN_PNG).is_file():
    print("\n=== Causal chain image (multi-file root -> sink) ===")
    display(Image(filename=str(CHAIN_PNG)))
else:
    print("\nNo chain PNG detected; check which path your inference script uses for figures.")
